In [7]:
# ==========================
# 1. Imports
# ==========================
from pathlib import Path
import platform

import duckdb
import pandas as pd

# ==========================
# 1. Import
# ==========================
import numpy as np


# ==========================
# 2. Parâmetros
# ==========================
GESTAO = "LIS"

ANOS = [
    2025,
    2026,
]


# ==========================
# 3. Pasta das bases
# ==========================
def get_db_dir():

    sistema = platform.system()

    if sistema == "Windows":
        return Path(
            r"C:\Users\LISARR\OneDrive - Salvesen Logística S.A\python\00.DB"
        )

    if sistema == "Darwin":
        return Path(
            "/Users/rr/Library/Mobile Documents/"
            "com~apple~CloudDocs/05.Salvesen/00_DB"
        )

    return Path.cwd()


DB_DIR = get_db_dir()


# ==========================
# 4. Caminhos
# ==========================
PATHS = {
    ano: DB_DIR / f"{ano}.duckdb"
    for ano in ANOS
}


# ==========================
# 5. Validar bases
# ==========================
for ano, path in PATHS.items():

    if not path.exists():
        raise FileNotFoundError(
            f"Base {ano} não encontrada: {path}"
        )


# ==========================
# 6. Ligações DuckDB
# ==========================
CON = {
    ano: duckdb.connect(
        str(PATHS[ano]),
        read_only=True
    )
    for ano in ANOS
}


# ==========================
# 7. Validar
# ==========================
validacao = pd.DataFrame({
    "Ano": ANOS,
    "Base": [
        PATHS[ano].name
        for ano in ANOS
    ],
    "Tabelas": [
        len(CON[ano].sql("SHOW TABLES").fetchall())
        for ano in ANOS
    ],
})

validacao

,Ano,Base,Tabelas
0,2025,2025.duckdb,9
1,2026,2026.duckdb,10


In [8]:
# ==========================
# 1. Configuração
# ==========================
meses = {
    1: "Jan",
    2: "Fev",
    3: "Mar",
    4: "Abr",
    5: "Mai",
    6: "Jun",
    7: "Jul",
    8: "Ago",
    9: "Set",
    10: "Out",
    11: "Nov",
    12: "Dez",
}

tabelas_paletes = {}


# ==========================
# 2. Processar anos
# ==========================
for ano in ANOS:

    df_ano = CON[ano].execute(f"""
        SELECT
            CODACT,
            ACTIVIDAD AS CLIENTE,
            FENTREGA,
            TRY_CAST(
                REPLACE(PALETS, ',', '.')
                AS DOUBLE
            ) AS PALETS
        FROM inform_27_{ano}
        WHERE Gestion = ?
    """, [GESTAO]).df()

    # ==========================
    # 3. Extrair mês
    # ==========================
    df_ano["MES"] = pd.to_numeric(
        df_ano["FENTREGA"]
        .astype("string")
        .str.strip()
        .str[4:6],
        errors="coerce"
    )

    # ==========================
    # 4. Criar tabela mensal
    # ==========================
    tabela = (
        df_ano
        .pivot_table(
            index=["CODACT", "CLIENTE"],
            columns="MES",
            values="PALETS",
            aggfunc="sum",
            fill_value=0
        )
        .reindex(
            columns=range(1, 13),
            fill_value=0
        )
        .rename(columns=meses)
        .reset_index()
    )

    # ==========================
    # 5. Formatar paletes
    # ==========================
    colunas_meses = list(meses.values())

    tabela[colunas_meses] = (
        tabela[colunas_meses]
        .round(0)
        .astype(int)
    )

    tabelas_paletes[ano] = tabela


# ==========================
# 6. Mostrar 2026
# ==========================
tabelas_paletes[2026]

MES,CODACT,CLIENTE,Jan,Fev,Mar,Abr,Mai,Jun,Jul,Ago,Set,Out,Nov,Dez
0,011,DANONE PORTUGAL,2939,2473,14378,15000,16625,15909,16358,18342,15923,1158,0,0
1,013,DANONE PORTUGAL CAPILAR,2982,2914,4166,3990,3609,3173,3365,3046,3207,263,0,0
2,015,TROPICANA,22,8,109,92,154,256,383,234,202,0,0,0
3,028,PLN,0,0,0,0,1,9,18,25,28,8,0,0
4,068,FRIESLANDCAMPIÑA,1,0,7,0,15,0,0,0,0,0,0,0
5,074,NUTRICIA,1298,858,5599,4934,4221,3778,4169,4181,4452,292,0,0
6,084,DANONE PORTUGAL TLD,495,363,528,495,528,1695,807,528,627,0,0,0
7,116,ORANGINA SCHWEPPES ESPAÑA BASE,0,0,0,0,3,6,5,8,6,0,0,0
8,117,ORANGINA SCHWEPPES,0,8,72,88,84,192,288,170,118,0,0,0
9,118,ORANGNA SCHWEPPES PORTUGAL,59,45,482,447,422,501,594,588,522,27,0,0


In [9]:
# ==========================
# 1. Configuração
# ==========================
ano = 2025

meses = {
    1: "Jan",
    2: "Fev",
    3: "Mar",
    4: "Abr",
    5: "Mai",
    6: "Jun",
    7: "Jul",
    8: "Ago",
    9: "Set",
    10: "Out",
    11: "Nov",
    12: "Dez",
}


# ==========================
# 2. Ler dados
# ==========================
df_2025 = CON[ano].execute(f"""
    SELECT
        CODACT,
        ACTIVIDAD AS CLIENTE,
        FENTREGA,
        TRY_CAST(
            REPLACE(PALETS, ',', '.')
            AS DOUBLE
        ) AS PALETS
    FROM inform_27_{ano}
    WHERE Gestion = ?
""", [GESTAO]).df()


# ==========================
# 3. Extrair mês
# ==========================
df_2025["MES"] = pd.to_numeric(
    df_2025["FENTREGA"]
    .astype("string")
    .str.strip()
    .str[4:6],
    errors="coerce"
)


# ==========================
# 4. Criar tabela mensal
# ==========================
tabela_paletes_2025 = (
    df_2025
    .pivot_table(
        index=["CODACT", "CLIENTE"],
        columns="MES",
        values="PALETS",
        aggfunc="sum",
        fill_value=0
    )
    .reindex(
        columns=range(1, 13),
        fill_value=0
    )
    .rename(columns=meses)
    .reset_index()
)


# ==========================
# 5. Formatar paletes
# ==========================
colunas_meses = list(meses.values())

tabela_paletes_2025[colunas_meses] = (
    tabela_paletes_2025[colunas_meses]
    .round(0)
    .astype(int)
)


# ==========================
# 6. Mostrar
# ==========================
tabela_paletes_2025

MES,CODACT,CLIENTE,Jan,Fev,Mar,Abr,Mai,Jun,Jul,Ago,Set,Out,Nov,Dez
0,011,DANONE PORTUGAL,7358,7457,6611,7720,7963,7816,9843,5863,2787,8690,8777,2943
1,013,DANONE PORTUGAL CAPILAR,1692,1550,1229,900,1574,1658,1793,877,1439,3211,3883,2649
2,015,TROPICANA,63,78,64,84,146,182,277,141,37,120,106,30
3,068,FRIESLANDCAMPIÑA,2,0,1,0,0,0,2,4,3,10,5,2
4,074,NUTRICIA,3294,2464,2770,2867,3583,3298,3990,1807,996,3247,3260,1464
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62,711,XPO HELM,93,108,58,45,54,88,41,19,58,80,0,0
63,757,SUMOLCOMPAL MARKETING,82,89,39,83,46,90,236,86,108,228,119,17
64,784,SABORES DO VEZ,40,45,40,33,77,37,49,16,53,29,30,34
65,860,ANDROS PTG CONGELADO,0,0,1,2,0,11,0,0,0,0,0,0


In [10]:
# ==========================
# 1. Validar cliente 011
# ==========================
validacao_011 = CON[2025].execute("""
    SELECT
        SUBSTRING(FENTREGA, 5, 2) AS MES,
        COUNT(*) AS LINHAS,
        COUNT(DISTINCT CODEUT) AS VIAGENS,
        COUNT(DISTINCT FENTREGA) AS DIAS,
        MIN(FENTREGA) AS PRIMEIRA_ENTREGA,
        MAX(FENTREGA) AS ULTIMA_ENTREGA,
        ROUND(
            SUM(
                TRY_CAST(
                    REPLACE(PALETS, ',', '.')
                    AS DOUBLE
                )
            ),
            0
        ) AS PALETS
    FROM inform_27_2025
    WHERE Gestion = 'LIS'
      AND CODACT = '011'
    GROUP BY
        SUBSTRING(FENTREGA, 5, 2)
    ORDER BY
        MES
""").df()


# ==========================
# 2. Mostrar
# ==========================
validacao_011

,MES,LINHAS,VIAGENS,DIAS,PRIMEIRA_ENTREGA,ULTIMA_ENTREGA,PALETS
0,01,1500,575,25,20250102,20250131,7358.0
1,02,1423,501,24,20250201,20250228,7457.0
2,03,1283,496,26,20250301,20250331,6611.0
3,04,1559,465,26,20250401,20250430,7720.0
4,05,1514,498,27,20250501,20250531,7963.0
5,06,1435,489,25,20250602,20250630,7816.0
6,07,2032,682,26,20250701,20250731,9843.0
7,08,1153,469,24,20250801,20250829,5863.0
8,09,713,273,23,20250901,20250930,2787.0
9,10,1557,624,27,20251001,20251031,8690.0


In [22]:
# ==========================
# 1. Dados reais de 2025
# ==========================
dados_2025 = CON[2025].execute("""
    SELECT
        TRIM(CAST(CODACT AS VARCHAR)) AS CODACT,
        MODE(ACTIVIDAD) AS CLIENTE,
        CAST(SUBSTRING(FENTREGA, 5, 2) AS INTEGER) AS MES,
        SUM(
            TRY_CAST(
                REPLACE(PALETS, ',', '.')
                AS DOUBLE
            )
        ) AS PALETS
    FROM inform_27_2025
    WHERE Gestion = 'LIS'
      AND FENTREGA BETWEEN '20250101' AND '20251231'
    GROUP BY
        CODACT,
        MES
""").df()


# ==========================
# 2. Tabela mensal 2025
# ==========================
MESES = {
    1: "Jan",
    2: "Fev",
    3: "Mar",
    4: "Abr",
    5: "Mai",
    6: "Jun",
    7: "Jul",
    8: "Ago",
    9: "Set",
    10: "Out",
    11: "Nov",
    12: "Dez",
}

tabela_2025 = (
    dados_2025
    .pivot_table(
        index=["CODACT", "CLIENTE"],
        columns="MES",
        values="PALETS",
        aggfunc="sum",
        fill_value=0
    )
    .reindex(
        columns=range(1, 13),
        fill_value=0
    )
    .rename(columns=MESES)
    .reset_index()
)

tabela_2025[list(MESES.values())] = (
    tabela_2025[list(MESES.values())]
    .round()
    .astype(int)
)

# ==========================
# 3. Mostrar completo
# ==========================
with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.width", None,
):
    display(tabela_2025)

MES,CODACT,CLIENTE,Jan,Fev,Mar,Abr,Mai,Jun,Jul,Ago,Set,Out,Nov,Dez
0,011,DANONE PORTUGAL,7358,7457,6611,7720,7963,7816,9843,5863,2787,8690,8777,2943
1,013,DANONE PORTUGAL CAPILAR,1692,1550,1229,900,1574,1658,1793,877,1439,3211,3883,2649
2,015,TROPICANA,63,78,64,84,146,182,277,141,37,120,106,30
3,068,FRIESLANDCAMPIÑA,2,0,1,0,0,0,2,4,3,10,5,2
4,074,NUTRICIA,3294,2464,2770,2867,3583,3298,3990,1807,996,3247,3260,1464
5,084,DANONE PORTUGAL TLD,264,66,132,66,297,396,759,462,792,423,429,528
6,116,ORANGINA SCHWEPPES ESPAÑA BASE,0,0,0,0,0,9,11,13,1,20,1,0
7,117,ORANGINA SCHWEPPES,35,0,0,13,18,33,106,64,23,72,54,23
8,118,ORANGNA SCHWEPPES PORTUGAL,189,107,147,128,172,220,400,247,46,349,229,103
9,131,RANA,746,638,611,591,518,563,908,550,312,1131,1054,373


In [23]:
# ==========================
# 1. Janeiro 2026
# ==========================
jan_2026 = CON[2026].execute("""
    SELECT
        TRIM(CAST(CODACT AS VARCHAR)) AS CODACT,
        SUM(
            TRY_CAST(
                REPLACE(PALETS, ',', '.')
                AS DOUBLE
            )
        ) AS Jan_2026
    FROM inform_27_2026
    WHERE Gestion = 'LIS'
      AND FENTREGA BETWEEN '20260101' AND '20260131'
    GROUP BY CODACT
""").df()


# ==========================
# 2. Normalizar CODACT
# ==========================
jan_2026["CODACT"] = (
    jan_2026["CODACT"]
    .apply(normalizar_codact)
)


# ==========================
# 3. Juntar Janeiro 2026
# ==========================
base_variacao = tabela_2025.merge(
    jan_2026,
    on="CODACT",
    how="left"
)

base_variacao["Jan_2026"] = (
    base_variacao["Jan_2026"]
    .fillna(0)
)


# ==========================
# 4. Criar tabela
# ==========================
variacao_2025 = base_variacao[
    ["CODACT", "CLIENTE"]
].copy()


# ==========================
# 5. Variações normais
# ==========================
pares = [
    ("Jan", "Fev"),
    ("Fev", "Mar"),
    ("Mar", "Abr"),
    ("Abr", "Mai"),
    ("Mai", "Jun"),
    ("Jun", "Jul"),
    ("Jul", "Ago"),
    ("Out", "Nov"),
]

for anterior, atual in pares:

    variacao_2025[f"{anterior}-{atual}"] = np.where(
        base_variacao[anterior] > 0,
        (
            base_variacao[atual]
            / base_variacao[anterior]
            - 1
        ) * 100,
        np.nan
    )


# ==========================
# 6. Agosto para Outubro
# ==========================
variacao_2025["Ago-Out"] = np.where(
    (base_variacao["Ago"] > 0)
    & (base_variacao["Out"] > 0),
    (
        np.sqrt(
            base_variacao["Out"]
            / base_variacao["Ago"]
        )
        - 1
    ) * 100,
    np.nan
)


# ==========================
# 7. Novembro para Janeiro
# ==========================
variacao_2025["Nov-Jan26"] = np.where(
    (base_variacao["Nov"] > 0)
    & (base_variacao["Jan_2026"] > 0),
    (
        np.sqrt(
            base_variacao["Jan_2026"]
            / base_variacao["Nov"]
        )
        - 1
    ) * 100,
    np.nan
)


# ==========================
# 8. Ordenar colunas
# ==========================
variacao_2025 = variacao_2025[
    [
        "CODACT",
        "CLIENTE",
        "Jan-Fev",
        "Fev-Mar",
        "Mar-Abr",
        "Abr-Mai",
        "Mai-Jun",
        "Jun-Jul",
        "Jul-Ago",
        "Ago-Out",
        "Out-Nov",
        "Nov-Jan26",
    ]
]


# ==========================
# 9. Formatar
# ==========================
colunas_variacao = variacao_2025.columns[2:]

variacao_2025[colunas_variacao] = (
    variacao_2025[colunas_variacao]
    .round(1)
)


# ==========================
# 10. Mostrar completo
# ==========================
with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.width", None,
):
    display(variacao_2025)

,CODACT,CLIENTE,Jan-Fev,Fev-Mar,Mar-Abr,Abr-Mai,Mai-Jun,Jun-Jul,Jul-Ago,Ago-Out,Out-Nov,Nov-Jan26
0,011,DANONE PORTUGAL,1.3,-11.3,16.8,3.1,-1.8,25.9,-40.4,21.7,1.0,NaN
1,013,DANONE PORTUGAL CAPILAR,-8.4,-20.7,-26.8,74.9,5.3,8.1,-51.1,91.3,20.9,NaN
2,015,TROPICANA,23.8,-17.9,31.2,73.8,24.7,52.2,-49.1,-7.7,-11.7,NaN
3,068,FRIESLANDCAMPIÑA,-100.0,NaN,-100.0,NaN,NaN,NaN,100.0,58.1,-50.0,NaN
4,074,NUTRICIA,-25.2,12.4,3.5,25.0,-8.0,21.0,-54.7,34.0,0.4,NaN
5,084,DANONE PORTUGAL TLD,-75.0,100.0,-50.0,350.0,33.3,91.7,-39.1,-4.3,1.4,NaN
6,116,ORANGINA SCHWEPPES ESPAÑA BASE,NaN,NaN,NaN,NaN,NaN,22.2,18.2,24.0,-95.0,NaN
7,117,ORANGINA SCHWEPPES,-100.0,NaN,NaN,38.5,83.3,221.2,-39.6,6.1,-25.0,NaN
8,118,ORANGNA SCHWEPPES PORTUGAL,-43.4,37.4,-12.9,34.4,27.9,81.8,-38.2,18.9,-34.4,-49.2
9,131,RANA,-14.5,-4.2,-3.3,-12.4,8.7,61.3,-39.4,43.4,-6.8,-42.9


In [24]:
# ==========================
# 1. Parâmetros
# ==========================
LIMITE_PALETES = 500

MESES_2026 = {
    1: "Jan",
    2: "Fev",
    3: "Mar",
    4: "Abr",
    5: "Mai",
    6: "Jun",
    7: "Jul",
    8: "Ago",
    9: "Set",
}


# ==========================
# 2. Dados reais 2026
# ==========================
dados_2026 = CON[2026].execute("""
    SELECT
        TRIM(CAST(CODACT AS VARCHAR)) AS CODACT,
        MODE(ACTIVIDAD) AS CLIENTE,
        CAST(SUBSTRING(FENTREGA, 5, 2) AS INTEGER) AS MES,
        SUM(
            TRY_CAST(
                REPLACE(PALETS, ',', '.')
                AS DOUBLE
            )
        ) AS PALETS
    FROM inform_27_2026
    WHERE Gestion = 'LIS'
      AND FENTREGA BETWEEN '20260101' AND '20260930'
    GROUP BY
        CODACT,
        MES
""").df()


# ==========================
# 3. Normalizar CODACT
# ==========================
dados_2026["CODACT"] = (
    dados_2026["CODACT"]
    .apply(normalizar_codact)
)


# ==========================
# 4. Tabela mensal 2026
# ==========================
tabela_2026 = (
    dados_2026
    .pivot_table(
        index=["CODACT", "CLIENTE"],
        columns="MES",
        values="PALETS",
        aggfunc="sum",
        fill_value=0
    )
    .reindex(
        columns=range(1, 10),
        fill_value=0
    )
    .rename(columns=MESES_2026)
    .reset_index()
)


# ==========================
# 5. Juntar variações
# ==========================
resultado = tabela_2026.merge(
    variacao_2025[
        [
            "CODACT",
            "Ago-Out",
            "Out-Nov",
            "Nov-Jan26",
        ]
    ],
    on="CODACT",
    how="left"
)


# ==========================
# 6. Prever Outubro
# ==========================
resultado["Out Prev."] = (
    resultado["Set"]
    * (
        1
        + resultado["Ago-Out"] / 100
    )
)


# ==========================
# 7. Prever Novembro
# ==========================
resultado["Nov Prev."] = (
    resultado["Out Prev."]
    * (
        1
        + resultado["Out-Nov"] / 100
    )
)


# ==========================
# 8. Prever Dezembro
# ==========================
resultado["Dez Prev."] = (
    resultado["Nov Prev."]
    * (
        1
        + resultado["Nov-Jan26"] / 100
    )
)


# ==========================
# 9. Excluir residuais
# ==========================
meses_reais = [
    "Jan",
    "Fev",
    "Mar",
    "Abr",
    "Mai",
    "Jun",
    "Jul",
    "Ago",
    "Set",
]

resultado["TOTAL_REAL"] = (
    resultado[meses_reais]
    .sum(axis=1)
)

resultado = resultado[
    resultado["TOTAL_REAL"] >= LIMITE_PALETES
].copy()


# ==========================
# 10. Selecionar colunas
# ==========================
resultado = resultado[
    [
        "CODACT",
        "CLIENTE",
        "Jan",
        "Fev",
        "Mar",
        "Abr",
        "Mai",
        "Jun",
        "Jul",
        "Ago",
        "Set",
        "Out Prev.",
        "Nov Prev.",
        "Dez Prev.",
        "TOTAL_REAL",
    ]
]


# ==========================
# 11. Formatar
# ==========================
colunas_valores = resultado.columns[2:-1]

resultado[colunas_valores] = (
    resultado[colunas_valores]
    .round()
    .astype("Int64")
)


# ==========================
# 12. Ordenar
# ==========================
resultado = (
    resultado
    .sort_values(
        "TOTAL_REAL",
        ascending=False
    )
    .drop(columns="TOTAL_REAL")
    .reset_index(drop=True)
)


# ==========================
# 13. Mostrar completo
# ==========================
with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.width", None,
):
    display(resultado)

,CODACT,CLIENTE,Jan,Fev,Mar,Abr,Mai,Jun,Jul,Ago,Set,Out Prev.,Nov Prev.,Dez Prev.
0,11,DANONE PORTUGAL,2939,2473,14378,15000,16625,15909,16358,18342,15923,<NA>,<NA>,<NA>
1,74,NUTRICIA,1298,858,5599,4934,4221,3778,4169,4181,4452,<NA>,<NA>,<NA>
2,13,DANONE PORTUGAL CAPILAR,2982,2914,4166,3990,3609,3173,3365,3046,3207,<NA>,<NA>,<NA>
3,311,ALPRO PORTUGAL,1191,982,3118,3676,3295,3180,3805,3970,3518,4633,4693,3234
4,260,ANDROS REFRIGERADO,689,538,2564,2506,2701,2047,2289,2009,2112,2830,2431,1585
5,429,H-3 Carnes,559,394,1477,1500,1636,1595,1760,1778,1529,1694,2062,1466
6,404,RTM,268,309,1448,1466,1370,1408,1555,1509,1527,2518,1982,1191
7,131,RANA,344,253,1259,1249,1383,1161,1260,1220,1302,1867,1740,994
8,398,CONFEITARIA CARLOS GONÇALVES LDA,272,150,869,1177,1478,1027,1311,1188,1378,1437,973,772
9,391,GL IMPORTAÇÃO E EXPORTAÇÃO,187,136,909,944,1067,1114,1189,1107,1203,1454,865,426
